# 04 Machine-Learning Models & Interpretability

**Owner:** Teddy (Technical & ML Lead)

**Checklist:**
- [x] Logistic Regression, Random Forest, XGBoost through the walk-forward evaluator (T4)
- [x] Hyperparameter tuning strictly on the validation window (2018–2019), seeds fixed (SEED=42)
- [ ] LSTM deep sequence model (T5 - stretch goal)
- [x] SHAP feature attribution and Random Forest MDI feature importance (T6)
- [x] Multi-seed stability audit (5 random seeds: 42, 100, 2024, 777, 999) (T7)

**Hard Rules Enforced:**
1. Chronological splitting only — no random k-fold splits.
2. Features never use future data; targets are purged across segment boundaries.
3. Scalers, imputers, and tree parameters fit on training data only.
4. Numbers and figures come from `reports/tables/` and `reports/figures/`.

## 1. Setup & Environment

In [ ]:
import sys; sys.path.insert(0, '..')   # execute from notebooks/ directory
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from src.eval.config import SEED, WINDOWS
from src.eval.metrics import summarise
from src.eval.walkforward import walk_forward
from src.features.build import BASE_FEATURES, load_features
from src.models.ml.logistic import LogisticForecaster
from src.models.ml.random_forest import RandomForestForecaster
from src.models.ml.xgboost_model import XGBoostForecaster
from src.viz.style import PALETTE, apply_style, save

apply_style()
print("Environment initialized. Fixed seed:", SEED)

## 2. Load Processed Features & Verify Absence of Leakage

In [ ]:
feats = load_features('../data/processed/features.parquet')
print(f"Total feature rows: {len(feats):,}")
print("Entities present:", sorted(feats['entity'].unique()))
feats.head()

## 3. Validation Performance Comparison (2018–2019 Window)

We evaluate how our ML models compare against simple baselines (Zero / Martingale, Historical Mean, Persistence) on the validation window.

In [ ]:
metrics_val = pd.read_csv('../reports/tables/metrics_val.csv')

# Display Market Proxy (h=1 and h=5)
market_val = metrics_val[metrics_val['entity_set'] == 'MARKET'][['horizon', 'model', 'rmse', 'mae', 'oos_r2', 'accuracy', 'auc', 'dm_stat', 'dm_p']]
market_val.sort_values(['horizon', 'rmse'])

In [ ]:
# Display Pooled Panel (All 13 Stocks, h=1 and h=5)
panel_val = metrics_val[metrics_val['entity_set'] == 'PANEL'][['horizon', 'model', 'rmse', 'mae', 'oos_r2', 'accuracy', 'auc', 'dm_stat', 'dm_p']]
panel_val.sort_values(['horizon', 'rmse'])

## 4. Visualizing Model Performance on Validation Window

In [ ]:
# Plot Out-of-Sample R2 and Directional Accuracy for Market Proxy (h=1)
sub = metrics_val[(metrics_val['entity_set'] == 'MARKET') & (metrics_val['horizon'] == 1)].dropna(subset=['oos_r2'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

colors = [PALETTE['ml'] if m in ('logistic', 'random_forest', 'xgboost') else PALETTE['baseline'] for m in sub['model']]
ax1.barh(sub['model'], sub['oos_r2'], color=colors)
ax1.axvline(0, color='grey', linestyle='--', alpha=0.7)
ax1.set_xlabel('Out-of-Sample $R^2$ (vs Zero Baseline)')
ax1.set_title('Validation Forecast Accuracy (Market, h=1)')

sub_dir = metrics_val[(metrics_val['entity_set'] == 'MARKET') & (metrics_val['horizon'] == 1)].dropna(subset=['accuracy'])
dir_colors = [PALETTE['ml'] if m in ('logistic', 'random_forest', 'xgboost') else PALETTE['baseline'] for m in sub_dir['model']]
ax2.barh(sub_dir['model'], sub_dir['accuracy'] * 100, color=dir_colors)
base_rate = sub_dir['base_rate_up'].iloc[0] * 100
ax2.axvline(base_rate, color=PALETTE['shock'], linestyle='--', label=f'Base Rate Up ({base_rate:.1f}%)')
ax2.set_xlabel('Directional Accuracy (%)')
ax2.set_title('Directional Hit Rate (Market, h=1)')
ax2.legend(loc='lower right')

plt.tight_layout()
save(fig, 'ml_val_comparison_notebook')
plt.show()

## 5. Interpretability: SHAP Attributions & Feature Importance (Task T6)

To understand *why* the machine-learning models generate their forecasts and satisfy requirement 10 of the assignment, we compute SHAP (SHapley Additive exPlanations) values on the fitted gradient boosted trees.

In [ ]:
mkt = feats[feats['entity'] == 'MARKET'].sort_index()
train_mkt = mkt[mkt.index < '2018-01-01'].dropna(subset=['y_ret_1'])
test_mkt = mkt[(mkt.index >= '2018-01-01') & (mkt.index <= '2019-12-31')].dropna(subset=['y_ret_1'])

xgb_model = XGBoostForecaster(random_state=SEED)
xgb_model.fit(train_mkt, horizon=1)

X_test = test_mkt[xgb_model.feature_cols_]
explainer = shap.TreeExplainer(xgb_model.regressor_)
shap_values = explainer(X_test)

fig, ax = plt.subplots(figsize=(8, 5))
shap.plots.beeswarm(shap_values, show=False)
plt.title('SHAP Feature Importance: XGBoost (Market Proxy h=1)', fontsize=11)
plt.tight_layout()
save(fig, 'ml_shap_notebook')
plt.show()

In [ ]:
# Random Forest Feature Importance (MDI)
rf_model = RandomForestForecaster(random_state=SEED)
rf_model.fit(train_mkt, horizon=1)

fig, ax = plt.subplots(figsize=(8, 4.5))
rf_model.feature_importances_.sort_values().plot(kind='barh', color=PALETTE['ml'], ax=ax)
ax.set_title('Random Forest MDI Feature Importance (Market Proxy h=1)', fontsize=11)
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
save(fig, 'ml_rf_importance_notebook')
plt.show()

## 6. Stability & Robustness Audit (Task T7)

We evaluate whether the reported ML performance is sensitive to the random initialization seed across 5 independent seeds (42, 100, 2024, 777, 999).

In [ ]:
audit_df = pd.read_csv('../reports/tables/ml_stability_audit.csv')
audit_summary = audit_df.groupby('model')[['rmse', 'oos_r2', 'accuracy', 'auc']].agg(['mean', 'std']).round(5)
audit_summary

## 7. Key Findings & Financial Interpretation for the Report

1. **Out-of-sample predictability**: On the Market Proxy ($h=1$), both Random Forest ($R^2 = +4.07\%$) and XGBoost ($R^2 = +4.74\%$) significantly outperform the martingale/zero baseline. XGBoost achieves a statistically significant Diebold-Mariano test statistic of $t = -2.066$ ($p = 0.039$).
2. **Directional Accuracy**: Logistic regression demonstrates strong classification ability (60.9% hit rate on Market $h=1$ vs base rate of 43.9%), indicating that direction is more forecastable than exact magnitude.
3. **Feature Drivers**: As shown by the SHAP analysis, rolling volatility ($v_{22}, v_5$) and short momentum ($m_5, r, l_1$) are the primary drivers of return predictability, consistent with financial literature on volatility clustering and frontier market momentum.
4. **Stability**: Cross-seed evaluation yields standard deviations below 0.001 on RMSE, verifying that results are not artifacts of seed selection.